# Knowledge Distillation

B-0 Teacher로 MobileNetV3-Large Student를 학습한다. 비교 기준은 `baseline.ipynb`의 KD 없는 Student이다.

## 1. B-0 Teacher 불러오기와 검증 재현

- 협업자가 Colab에서 학습한 B-0 Teacher 체크포인트를 이 컴퓨터에서 불러온다.
- 같은 검증 데이터에서 정확도가 기록된 값(0.8333)과 같은지 확인한다. 다르면 구조 복원이나 전처리에 문제가 있으므로 KD를 시작하지 않는다.
- Teacher 체크포인트는 `outputs/weights/b0_teacher/best.pth`에 두며 Git에 올리지 않는다.

In [ ]:
# 공통 데이터 모듈을 프로젝트 루트에서 불러오고 검증 DataLoader를 만든다.
from pathlib import Path
import sys
import torch
from torch import nn

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from data_loader import (
    read_cub_metadata,
    split_cub_images,
    build_transforms,
    build_dataloaders,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

data_dir = project_root / "data" / "CUB_200_2011" / "CUB_200_2011"
image_list, image_labels, class_names = read_cub_metadata(data_dir)
train_image_list, val_image_list, test_image_list = split_cub_images(
    data_dir, image_list, image_labels
)
train_transform, eval_transform = build_transforms()

train_loader, val_loader, test_loader = build_dataloaders(
    data_dir=data_dir,
    image_labels=image_labels,
    train_image_list=train_image_list,
    val_image_list=val_image_list,
    test_image_list=test_image_list,
    train_transform=train_transform,
    eval_transform=eval_transform,
    batch_size=32,
    num_workers=0,
)

# 분할이 공유 기준인 docs/splits의 ID 목록과 같은지 확인한다.
for name, rows in [("train", train_image_list), ("validation", val_image_list), ("test", test_image_list)]:
    saved_ids = (project_root / "docs" / "splits" / f"{name}_ids.txt").read_text().split()
    print(name, "일치:", [row.split()[0] for row in rows] == saved_ids, "개수:", len(rows))
print("device:", device, "| torch:", torch.__version__)

In [ ]:
# Teacher 구조를 만들기 위한 DINOv3 저장소의 커밋을 확인한다.
import subprocess

dinov3_dir = project_root / "outputs" / "external" / "dinov3"
expected_commit = "6876159a11b4df116f30f667f8c9888617df0751"  # 협업자의 학습 노트북에 기록된 커밋

commit = subprocess.run(
    ["git", "-C", str(dinov3_dir), "rev-parse", "HEAD"],
    capture_output=True, text=True,
).stdout.strip()

print("DINOv3 커밋:", commit)
print("협업자와 일치:", commit == expected_commit)

In [ ]:
# DINOv3 ConvNeXt-Base 구조를 만들고 200개 클래스 분류층을 붙인다.
if str(dinov3_dir) not in sys.path:
    sys.path.insert(0, str(dinov3_dir))

from dinov3.hub.backbones import dinov3_convnext_base

teacher = dinov3_convnext_base(pretrained=False)
teacher.head = nn.Linear(teacher.embed_dim, 200)

print("특징 벡터 크기:", teacher.embed_dim)
print("분류층:", teacher.head)

In [ ]:
# 체크포인트를 안전 모드로 불러와 Teacher에 복원한다.
checkpoint = torch.load(
    project_root / "outputs" / "weights" / "b0_teacher" / "best.pth",
    map_location="cpu",
    weights_only=True,
)

print("체크포인트 epoch:", checkpoint["epoch"])
print("기록된 검증 정확도:", checkpoint["validation_accuracy"])
print("기록된 검증 loss:", checkpoint["validation_loss"])

print(teacher.load_state_dict(checkpoint["model_state_dict"], strict=True))

teacher = teacher.to(device)
teacher.eval()

# KD에서 Teacher는 학습하지 않으므로 기울기 계산을 끈다.
for parameter in teacher.parameters():
    parameter.requires_grad = False

In [ ]:
# 데이터 전체의 평균 loss와 정확도를 계산한다. 모든 계산은 float32로 수행한다.
criterion = nn.CrossEntropyLoss()

def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_count = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            batch_loss = criterion(outputs, labels)

            total_loss += batch_loss.item() * len(labels)
            total_correct += (outputs.argmax(dim=1) == labels).sum().item()
            total_count += len(labels)

    return total_loss / total_count, total_correct / total_count

teacher_loss, teacher_acc = evaluate(teacher, val_loader, criterion, device)

print(f"재현한 검증 loss: {teacher_loss:.4f}  (기록 {checkpoint['validation_loss']:.4f})")
print(f"재현한 검증 정확도: {teacher_acc:.4f}  (기록 {checkpoint['validation_accuracy']:.4f})")
print(f"정확도 차이: {abs(teacher_acc - checkpoint['validation_accuracy']):.4f}")

In [ ]:
# Teacher와 Student의 크기를 비교한다. Student는 baseline.ipynb에서 저장한 체크포인트를 사용한다.
from torchvision import models

student = models.mobilenet_v3_large(weights=None)
student.classifier[3] = nn.Linear(student.classifier[3].in_features, 200)
baseline_checkpoint = torch.load(project_root / "outputs" / "baseline_best.pt", map_location="cpu", weights_only=True)
student.load_state_dict(baseline_checkpoint["model_state"])
student = student.to(device)

teacher_params = sum(p.numel() for p in teacher.parameters())
student_params = sum(p.numel() for p in student.parameters())

print(f"Teacher 파라미터: {teacher_params:,}")
print(f"Student 파라미터: {student_params:,}")
print(f"크기 비율: Teacher가 Student의 {teacher_params / student_params:.1f}배")

In [ ]:
# 같은 검증 데이터에서 Teacher와 Student가 각각 맞히고 틀린 이미지를 센다.
teacher_correct = []
student_correct = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        labels = labels.to(device)
        teacher_correct.append(teacher(images).argmax(dim=1) == labels)
        student.eval()
        student_correct.append(student(images).argmax(dim=1) == labels)

teacher_correct = torch.cat(teacher_correct)
student_correct = torch.cat(student_correct)

total = len(teacher_correct)
print(f"Teacher만 맞힘: {(teacher_correct & ~student_correct).sum().item()}장 ({(teacher_correct & ~student_correct).float().mean().item():.1%})")
print(f"Student만 맞힘: {(~teacher_correct & student_correct).sum().item()}장 ({(~teacher_correct & student_correct).float().mean().item():.1%})")
print(f"둘 다 맞힘: {(teacher_correct & student_correct).sum().item()}장")
print(f"둘 다 틀림: {(~teacher_correct & ~student_correct).sum().item()}장")

## 2. 온도와 KD loss

- Teacher의 logits에 softmax를 적용하면 확률이 되지만, 보통 정답 클래스 하나가 거의 1에 가까워 다른 클래스의 정보가 사라진다.
- 온도 `T`로 logits를 나누고 softmax를 적용하면 확률이 부드러워져, "정답은 아니지만 비슷한 클래스"의 정보가 드러난다.
- KD loss는 정답 CrossEntropy와 Teacher 분포를 따라가는 KL divergence를 `alpha`로 섞는다.
- 아래 `T`와 `alpha`는 연습용 값이며, 본 실험의 값은 팀에서 정한 뒤 기록한다.

In [ ]:
# 검증 배치 하나에서 Teacher와 Student의 logits를 구한다.
import torch.nn.functional as F

batch_images, batch_labels = next(iter(val_loader))
batch_images = batch_images.to(device)
batch_labels = batch_labels.to(device)

with torch.no_grad():
    teacher_logits = teacher(batch_images)
    student_logits = student(batch_images)

print("Teacher logits:", teacher_logits.shape)
print("Student logits:", student_logits.shape)
print("Teacher 정확도(이 배치):", (teacher_logits.argmax(1) == batch_labels).float().mean().item())
print("Student 정확도(이 배치):", (student_logits.argmax(1) == batch_labels).float().mean().item())

In [ ]:
# Teacher는 맞히고 Student는 틀린 이미지 한 장에서 온도에 따른 확률 분포를 비교한다.
wrong_for_student = (teacher_logits.argmax(1) == batch_labels) & (student_logits.argmax(1) != batch_labels)
sample_index = wrong_for_student.nonzero()[0].item()
true_label = batch_labels[sample_index].item()
print("정답 클래스:", class_names[true_label])

for T in [1, 4]:
    probs = F.softmax(teacher_logits[sample_index] / T, dim=0)
    top_probs, top_classes = probs.topk(5)
    print(f"\n[Teacher, T={T}] 상위 5개 클래스")
    for prob, class_id in zip(top_probs.tolist(), top_classes.tolist()):
        mark = " <- 정답" if class_id == true_label else ""
        print(f"  {prob:6.3f}  {class_names[class_id]}{mark}")

student_probs = F.softmax(student_logits[sample_index], dim=0)
student_top, student_classes = student_probs.topk(3)
print("\n[Student, T=1] 상위 3개 클래스")
for prob, class_id in zip(student_top.tolist(), student_classes.tolist()):
    mark = " <- 정답" if class_id == true_label else ""
    print(f"  {prob:6.3f}  {class_names[class_id]}{mark}")

In [ ]:
# 온도에 따라 Teacher 확률이 얼마나 부드러워지는지 배치 전체의 평균으로 확인한다.
# 엔트로피가 클수록 확률이 여러 클래스에 퍼져 있다. 200개 클래스의 최대값은 ln(200) = 5.30이다.
print(f"{'T':>3} | {'최대 확률 평균':>12} | {'엔트로피 평균':>12}")
for T in [1, 2, 4, 8]:
    probs = F.softmax(teacher_logits / T, dim=1)
    max_prob = probs.max(dim=1).values.mean().item()
    entropy = -(probs * torch.log(probs)).sum(dim=1).mean().item()
    print(f"{T:>3} | {max_prob:>12.4f} | {entropy:>12.4f}")

In [ ]:
# KD loss를 계산한다. KL 항에는 T의 제곱을 곱해 온도에 따른 기울기 크기 변화를 보정한다.
T = 4
alpha = 0.5

ce_loss = F.cross_entropy(student_logits, batch_labels)
kd_loss = F.kl_div(
    F.log_softmax(student_logits / T, dim=1),
    F.softmax(teacher_logits / T, dim=1),
    reduction="batchmean",
) * (T * T)
loss = alpha * ce_loss + (1 - alpha) * kd_loss

print(f"CE 항: {ce_loss.item():.4f}")
print(f"KD 항(KL x T^2): {kd_loss.item():.4f}")
print(f"합산 loss (alpha={alpha}): {loss.item():.4f}")

In [ ]:
# KL divergence를 정의대로 직접 계산하여 F.kl_div와 비교한다.
teacher_probs = F.softmax(teacher_logits / T, dim=1)
log_teacher = F.log_softmax(teacher_logits / T, dim=1)
log_student = F.log_softmax(student_logits / T, dim=1)

manual_kl = (teacher_probs * (log_teacher - log_student)).sum(dim=1).mean() * (T * T)

print("직접 계산한 KD 항:", manual_kl.item())
print("일치:", torch.isclose(manual_kl, kd_loss).item())

# Student가 Teacher와 같은 logits를 내면 KL은 0이어야 한다.
same_kl = F.kl_div(
    F.log_softmax(teacher_logits / T, dim=1),
    F.softmax(teacher_logits / T, dim=1),
    reduction="batchmean",
)
print("Student가 Teacher와 같을 때 KL:", same_kl.item())

In [ ]:
# T^2를 곱하지 않으면 온도가 커질수록 KL 항의 기울기가 작아지는 것을 확인한다.
print(f"{'T':>3} | {'T^2 없음':>10} | {'T^2 곱함':>10}   (Student logits에 대한 기울기 크기)")
for T_value in [1, 2, 4, 8]:
    logits = student_logits.clone().requires_grad_(True)
    kl = F.kl_div(
        F.log_softmax(logits / T_value, dim=1),
        F.softmax(teacher_logits / T_value, dim=1),
        reduction="batchmean",
    )
    grad_plain = torch.autograd.grad(kl, logits)[0].norm().item()
    print(f"{T_value:>3} | {grad_plain:>10.5f} | {grad_plain * T_value ** 2:>10.5f}")

## 3. B-0 KD 학습

- Baseline과 같은 seed, 같은 초기화, 같은 학습 설정으로 Student를 학습하고 loss만 KD loss로 바꾼다. 차이가 KD에서만 나오도록 하기 위해서다.
- Teacher는 `eval()`과 `no_grad`로 추론만 하며 학습하지 않는다.
- 매 epoch CE 항과 KD 항을 따로 기록한다. KD loss가 줄었다는 것만으로 성공을 판단하지 않고 검증 정확도를 함께 본다.
- `T`와 `alpha`는 연습용 값이며, 본 실험의 값은 팀에서 정한 뒤 기록한다.

In [ ]:
# 학습 설정을 정한다. KD를 제외한 값은 baseline.ipynb와 같다.
kd_config = {
    "seed": 0,
    "epochs": 20,
    "batch_size": 32,
    "optimizer": "AdamW",
    "lr": 1e-4,
    "weight_decay": 0.01,
    "student_weights": "MobileNet_V3_Large_Weights.IMAGENET1K_V2",
    "teacher_checkpoint": "B-0 best.pth (epoch 10)",
    "T": 4,
    "alpha": 0.5,
    "torch": str(torch.__version__),
}

# 학습에 쓰이는 난수를 고정하는 함수이다. baseline.ipynb와 같은 방식이다.
import random

def set_seed(seed):
    random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Baseline과 같은 순서로 seed를 고정하고 Student를 만든다.
set_seed(kd_config["seed"])

kd_student = models.mobilenet_v3_large(weights=models.MobileNet_V3_Large_Weights.IMAGENET1K_V2)
kd_student.classifier[3] = nn.Linear(kd_student.classifier[3].in_features, 200)
kd_student = kd_student.to(device)

kd_optimizer = torch.optim.AdamW(
    kd_student.parameters(), lr=kd_config["lr"], weight_decay=kd_config["weight_decay"]
)

# Teacher는 학습하지 않으며 평가 모드여야 한다.
assert not teacher.training
assert all(not p.requires_grad for p in teacher.parameters())
print("학습 대상 파라미터(Student):", sum(p.numel() for p in kd_student.parameters() if p.requires_grad))

In [ ]:
# KD loss와 한 epoch 학습 함수를 정의한다.
def kd_loss_fn(student_logits, teacher_logits, labels, T, alpha):
    ce_loss = F.cross_entropy(student_logits, labels)
    kd_loss = F.kl_div(
        F.log_softmax(student_logits / T, dim=1),
        F.softmax(teacher_logits / T, dim=1),
        reduction="batchmean",
    ) * (T * T)
    return alpha * ce_loss + (1 - alpha) * kd_loss, ce_loss, kd_loss

def train_one_epoch_kd(student, teacher, loader, optimizer, device, T, alpha):
    student.train()
    total = {"loss": 0.0, "ce": 0.0, "kd": 0.0, "correct": 0, "count": 0}

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        # Teacher 출력은 기울기 계산 없이 구한다.
        with torch.no_grad():
            teacher_logits = teacher(images)

        student_logits = student(images)
        loss, ce_loss, kd_loss = kd_loss_fn(student_logits, teacher_logits, labels, T, alpha)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        n = len(labels)
        total["loss"] += loss.item() * n
        total["ce"] += ce_loss.item() * n
        total["kd"] += kd_loss.item() * n
        total["correct"] += (student_logits.argmax(dim=1) == labels).sum().item()
        total["count"] += n

    count = total["count"]
    return total["loss"] / count, total["ce"] / count, total["kd"] / count, total["correct"] / count

In [ ]:
# epoch마다 KD 학습과 검증을 수행하고 검증 정확도가 최고일 때 체크포인트를 저장한다.
import time
import json

kd_checkpoint_path = project_root / "outputs" / "kd_b0_best.pt"
kd_history = []
kd_best_val_acc = 0.0

for epoch in range(1, kd_config["epochs"] + 1):
    start = time.time()
    loss, ce_loss, kd_loss, train_acc = train_one_epoch_kd(
        kd_student, teacher, train_loader, kd_optimizer, device, kd_config["T"], kd_config["alpha"]
    )
    val_loss, val_acc = evaluate(kd_student, val_loader, criterion, device)
    kd_history.append({
        "epoch": epoch, "train_loss": loss, "train_ce": ce_loss, "train_kd": kd_loss,
        "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc,
    })

    saved = ""
    if val_acc > kd_best_val_acc:
        kd_best_val_acc = val_acc
        torch.save({
            "model_state": kd_student.state_dict(),
            "epoch": epoch, "val_acc": val_acc, "val_loss": val_loss, "config": kd_config,
        }, kd_checkpoint_path)
        saved = " 저장"

    print(
        f"epoch {epoch:2d} | CE {ce_loss:.3f} KD {kd_loss:.3f} | train acc {train_acc:.4f} | "
        f"val loss {val_loss:.4f} acc {val_acc:.4f} | {time.time() - start:.0f}초{saved}"
    )

with open(project_root / "outputs" / "kd_b0_history.json", "w") as f:
    json.dump({"config": kd_config, "history": kd_history}, f, ensure_ascii=False, indent=2)

In [ ]:
# KD 학습 곡선을 Baseline과 같은 그래프에서 비교한다.
import matplotlib.pyplot as plt

baseline_history = json.load(open(project_root / "outputs" / "baseline_history.json"))["history"]

epochs = [row["epoch"] for row in kd_history]
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs, [row["train_ce"] for row in kd_history], label="CE 항")
axes[0].plot(epochs, [row["train_kd"] for row in kd_history], label="KD 항")
axes[0].set_title("KD 학습 loss 항")
axes[0].set_xlabel("epoch")
axes[0].legend()

axes[1].plot(epochs, [row["val_loss"] for row in baseline_history], label="Baseline")
axes[1].plot(epochs, [row["val_loss"] for row in kd_history], label="B-0 KD")
axes[1].set_title("검증 loss(CE)")
axes[1].set_xlabel("epoch")
axes[1].legend()

axes[2].plot(epochs, [row["val_acc"] for row in baseline_history], label="Baseline")
axes[2].plot(epochs, [row["val_acc"] for row in kd_history], label="B-0 KD")
axes[2].axhline(checkpoint["validation_accuracy"], color="gray", linestyle="--", label="Teacher")
axes[2].set_title("검증 정확도")
axes[2].set_xlabel("epoch")
axes[2].legend()

fig.tight_layout()
plt.show()

In [ ]:
# Baseline과 KD Student의 검증 결과를 비교한다. 단일 seed이므로 잠정 비교이다.
def best_row(history):
    return max(history, key=lambda row: row["val_acc"])

base_best = best_row(baseline_history)
kd_best = best_row(kd_history)

print(f"{'조건':<10} | {'최고 검증 정확도':>14} | {'epoch':>5} | {'마지막 epoch 정확도':>16}")
print(f"{'Baseline':<10} | {base_best['val_acc']:>14.4f} | {base_best['epoch']:>5} | {baseline_history[-1]['val_acc']:>16.4f}")
print(f"{'B-0 KD':<10} | {kd_best['val_acc']:>14.4f} | {kd_best['epoch']:>5} | {kd_history[-1]['val_acc']:>16.4f}")
print(f"{'Teacher':<10} | {checkpoint['validation_accuracy']:>14.4f} | {checkpoint['epoch']:>5} |")

print()
print(f"KD Gain(최고 기준): {(kd_best['val_acc'] - base_best['val_acc']) * 100:+.2f}%p")
print(f"KD Gain(마지막 epoch 기준): {(kd_history[-1]['val_acc'] - baseline_history[-1]['val_acc']) * 100:+.2f}%p")